**Purpose:** Aggregate clean Silver data into business-ready Gold Delta tables.

- Gold table 1 — hourly aggregations per machine and sensor
- Gold table 2 — daily machine health summary
- Gold table 3 — anomaly detection using window functions

The Gold layer is the **final, business-ready** layer. It is NOT raw data
and NOT just cleaned data — it is **aggregated, enriched, and shaped**
for a specific consumer.

| Consumer | What they need | Gold table |
|---|---|---|
| Operations dashboard | Hourly avg per sensor per machine | `gold_hourly_stats` |
| Maintenance team | Daily machine health score | `gold_daily_health` |
| Alert system | Anomalous readings flagged | `gold_anomalies` |

Each Gold table answers one business question. This is important —
Gold tables are NOT general purpose. If a new team needs a different
view of the data, you build them a new Gold table, not modify an existing one

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG_NAME = "iot_pipeline"
SCHEMA_NAME  = "sensors"

SILVER_TABLE     = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_sensors"
GOLD_HOURLY      = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_hourly_stats"
GOLD_DAILY       = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_daily_health"
GOLD_ANOMALIES   = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_anomalies"

print(f"Silver source : {SILVER_TABLE}")
print(f"Gold hourly   : {GOLD_HOURLY}")
print(f"Gold daily    : {GOLD_DAILY}")
print(f"Gold anomalies: {GOLD_ANOMALIES}")

In [0]:

# ============================================================
# verifying the source table before building on top
# of it. If Silver has a problem, your Gold tables inherit it.
# This is a lightweight check — no .count() here, we use
# Delta metadata instead.
# ============================================================
df_silver = spark.table(SILVER_TABLE)

# Lightweight schema check — no full scan
print("Silver schema")
df_silver.printSchema()

# Use Delta metadata for row count — no full table scan
spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}") \
     .select("numFiles", "sizeInBytes") \
     .show()

# Quick sample to visually confirm the data looks right
print("Silver sample ")
display(df_silver.limit(5))

## Gold Table 1 — Hourly stats per machine and sensor

**Business question:** "For each machine, what were the average, min,
and max sensor readings every hour?"

**Consumer:** Operations dashboard — updates every hour,
shows engineers if any machine is trending toward a fault.

In [0]:
df_gold_hourly = spark.sql(f"""
    SELECT
        machine_id,
        sensor_type,
        unit,
        date_trunc('hour', timestamp)   AS hour_window,
        COUNT(*)                         AS reading_count,
        ROUND(AVG(value), 4)             AS avg_value,
        ROUND(MIN(value), 4)             AS min_value,
        ROUND(MAX(value), 4)             AS max_value,
        ROUND(STDDEV(value), 4)          AS stddev_value
    FROM {SILVER_TABLE}
    GROUP BY
        machine_id,
        sensor_type,
        unit,
        date_trunc('hour', timestamp)
    ORDER BY
        machine_id,
        sensor_type,
        hour_window
""")

print(f"Hourly stats rows: {df_gold_hourly.count():,}")
display(df_gold_hourly.limit(20))

- Partitioning by date means those queries only open the
- relevant date folders. A query for today doesn't touch
- last month's files at all.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {GOLD_HOURLY}")

(
    df_gold_hourly
    .withColumn("date", F.to_date("hour_window"))  # partition column
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("date")
    .saveAsTable(GOLD_HOURLY)
)

print(f"Saved: {GOLD_HOURLY}")

## Gold Table 2 — Daily machine health summary

**Business question:** "How healthy was each machine today —
was it running within normal parameters for most of the day?"

**Consumer:** Maintenance team — they use this to plan
preventive maintenance before machines actually fail

# Defining a reading as "healthy" if it is within the valid
# range for its sensor type. The health score is the percentage
# of healthy readings across all sensors for that machine that day.
# Score of 100 = all readings normal.
# Score of 80 = 20% of readings were out of range — warning.
# Score below 70 = machine needs attention.

In [0]:
df_gold_daily = spark.sql(f"""
    SELECT
        machine_id,
        TO_DATE(timestamp)                          AS date,
        COUNT(*)                                     AS total_readings,
        COUNT(DISTINCT sensor_type)                  AS active_sensors,

        -- Health score: % of readings within valid range
        ROUND(
            AVG(CASE
                WHEN sensor_type = 'temperature' AND value BETWEEN 60.0 AND 120.0 THEN 1.0
                WHEN sensor_type = 'pressure'    AND value BETWEEN 1.0  AND 10.0  THEN 1.0
                WHEN sensor_type = 'vibration'   AND value BETWEEN 0.01 AND 2.0   THEN 1.0
                ELSE 0.0
            END) * 100
        , 1)                                         AS health_score_pct,

        -- Average readings per sensor type for the day
        ROUND(AVG(CASE WHEN sensor_type = 'temperature' THEN value END), 2) AS avg_temperature,
        ROUND(AVG(CASE WHEN sensor_type = 'pressure'    THEN value END), 2) AS avg_pressure,
        ROUND(AVG(CASE WHEN sensor_type = 'vibration'   THEN value END), 2) AS avg_vibration,

        -- Flag machines that need attention
        CASE
            WHEN AVG(CASE
                WHEN sensor_type = 'temperature' AND value BETWEEN 60.0 AND 120.0 THEN 1.0
                WHEN sensor_type = 'pressure'    AND value BETWEEN 1.0  AND 10.0  THEN 1.0
                WHEN sensor_type = 'vibration'   AND value BETWEEN 0.01 AND 2.0   THEN 1.0
                ELSE 0.0
            END) * 100 >= 95 THEN 'NORMAL'
            WHEN AVG(CASE
                WHEN sensor_type = 'temperature' AND value BETWEEN 60.0 AND 120.0 THEN 1.0
                WHEN sensor_type = 'pressure'    AND value BETWEEN 1.0  AND 10.0  THEN 1.0
                WHEN sensor_type = 'vibration'   AND value BETWEEN 0.01 AND 2.0   THEN 1.0
                ELSE 0.0
            END) * 100 >= 80 THEN 'WARNING'
            ELSE 'CRITICAL'
        END                                          AS machine_status

    FROM {SILVER_TABLE}
    GROUP BY machine_id, TO_DATE(timestamp)
    ORDER BY date, machine_id
""")

print(f"Daily health rows: {df_gold_daily.count():,}")
display(df_gold_daily)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {GOLD_DAILY}")

(
    df_gold_daily
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("date")
    .saveAsTable(GOLD_DAILY)
)

print(f"Saved: {GOLD_DAILY}")

## Gold Table 3 — Anomaly detection with window functions

**Business question:** "Which individual sensor readings were
statistically abnormal compared to that machine's recent history?"

**Consumer:** Alert system — triggers a notification when a
machine's reading deviates significantly from its own baseline.

**Key Spark concept introduced:** Window functions — the most
powerful and most asked-about feature in Spark SQL interviews.

In [0]:
# Defining the window: per machine + sensor, ordered by time,
# looking at the last 50 readings
window_spec = (
    Window
    .partitionBy("machine_id", "sensor_type")
    .orderBy("timestamp")
    .rowsBetween(-50, 0)  # rolling window of last 50 readings
)

# Applying window functions to Silver data using PySpark
# (we use PySpark here because we're building the window spec
# programmatically — cleaner than raw SQL for this)
df_with_stats = (
    df_silver
    .withColumn("rolling_avg",    F.round(F.avg("value").over(window_spec), 4))
    .withColumn("rolling_stddev", F.round(F.stddev("value").over(window_spec), 4))

    # Z-score: how many standard deviations is this reading
    # from the rolling average?
    # z = (value - mean) / stddev
    # |z| > 2 means the reading is more than 2 stddevs away
    # from normal — statistically unusual (~5% of normal data)
    .withColumn("z_score", F.round(
        F.abs(F.col("value") - F.col("rolling_avg")) /
        F.when(F.col("rolling_stddev") > 0, F.col("rolling_stddev"))
         .otherwise(F.lit(1.0)),   # avoid division by zero
        4
    ))
    .withColumn("is_anomaly", F.col("z_score") > 2.0)
)

print(" Sample with rolling stats and z-score ")
display(
    df_with_stats
    .select("machine_id", "sensor_type", "timestamp",
            "value", "rolling_avg", "rolling_stddev",
            "z_score", "is_anomaly")
    .filter(F.col("is_anomaly") == True)
    .limit(20)
)

In [0]:
# registering the DataFrame as a temp view so Spark SQL
# can query it like a table. Temp views only exist for the
# duration of the Spark session — they are not saved to
# Unity Catalog. They're perfect for intermediate steps.
# ============================================================

# Register as a temp view so we can query it with SQL
df_with_stats.createOrReplaceTempView("silver_with_anomalies")

# Summarise anomalies per machine and sensor
anomaly_summary = spark.sql("""
    SELECT
        machine_id,
        sensor_type,
        COUNT(*)                                    AS total_readings,
        SUM(CAST(is_anomaly AS INT))                AS anomaly_count,
        ROUND(
            SUM(CAST(is_anomaly AS INT)) /
            COUNT(*) * 100
        , 2)                                        AS anomaly_rate_pct,
        ROUND(MAX(CASE WHEN is_anomaly THEN z_score END), 2) AS max_z_score
    FROM silver_with_anomalies
    GROUP BY machine_id, sensor_type
    ORDER BY anomaly_rate_pct DESC
""")

print("Anomaly summary per machine and sensor")
display(anomaly_summary)

In [0]:
# ============================================================
# We save only the anomalous rows to the Gold anomaly table
# — not all rows. The alert system only needs to know about
# problems, not every normal reading. This keeps the table
# small and fast to query.
#
# In a real pipeline, inserting into this table would
# trigger a downstream alert (email, Slack, PagerDuty).
# ============================================================

df_gold_anomalies = df_with_stats.filter(F.col("is_anomaly") == True)

spark.sql(f"DROP TABLE IF EXISTS {GOLD_ANOMALIES}")

(
    df_gold_anomalies
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("machine_id", "sensor_type")
    .saveAsTable(GOLD_ANOMALIES)
)

anomaly_count = df_gold_anomalies.count()
print(f"Saved: {GOLD_ANOMALIES}")
print(f"Total anomalies flagged: {anomaly_count:,}")

In [0]:
# ============================================================
# verifying all tables together:
# ============================================================

print("=" * 55)
print("  PIPELINE SUMMARY — ALL LAYERS")
print("=" * 55)

tables = {
    "Silver (source)"   : SILVER_TABLE,
    "Gold hourly stats" : GOLD_HOURLY,
    "Gold daily health" : GOLD_DAILY,
    "Gold anomalies"    : GOLD_ANOMALIES,
}

for label, table in tables.items():
    count = spark.sql(f"SELECT COUNT(*) AS n FROM {table}").collect()[0]["n"]
    print(f"  {label:<22}: {count:>6,} rows")

print("=" * 55)

# Quick spot check on Gold daily — confirm health scores look sensible
print("\n=== Machine health scores (all days averaged) ===")
spark.sql(f"""
    SELECT
        machine_id,
        ROUND(AVG(health_score_pct), 1) AS avg_health_score,
        MIN(machine_status)             AS worst_status
    FROM {GOLD_DAILY}
    GROUP BY machine_id
    ORDER BY avg_health_score DESC
""").show()